# BCT Hackathon User Modelling 
> Version : 5

## Goal 
Build an agent that understands users deeply enough to simulate their reviews — capturing tone, rating behaviour, and contextual nuance.
- Simulate star ratings and written reviews for 
unseen items
- Leverage user history, item metadata, and 
contextual signals
- Evaluated on review quality, rating accuracy, 
and behavioural fidelity

## Notebook version 2 
This is the final version of the first build of the it entails downloading the dataset from hugging face (we used the Kaggle beauty dataset first )
-  perfromed some much needed Exploratory Data Analysis on the data.
-  we created a function to clean the data into review rich data >10 or 100 words reviews
-  we used a hardcoded-review persona builder it builds persona on the reviewer using thier reviews
-  finally i used gemini 2.5 to generate reviews
-  then i  evaluated the data

## Notebook version 3 
- added a multi-agent workflow one agent builds the persona another builds the reviews
- used pydantic to structure the output of the agents so the agents can give us what we want and not unwanted stuff
- changed the evaluation pipeline (now we split the data into train, val and test with ratio 8:1:1)
- added techniques to save tokens 
- added something that limits character generation the review generated must be within the average character length of the reviewer (dont write more words than a reviewer will write)

## Notebook version 4 
- added rag to ground agent 2 reviews it didnt get the way the person was writing his reviews but with rag it will be able to have an example to build on (we will retireve reviews of products similar the one we want to generate for else we will retrieve the last 3 reviews of the user)

## Notebook version 5 (Here now)
- added cross-domain data (video game data of the amazon reviews data )
- added item enrichment (this uses the google search tool of the AI Agent 'gemini in this case ' to find out about the product to be reviewed and then returns it back into the agent 2 for more better reviews because the description text is mostly sales pitch and most times its noting)
- added  Rating before text generation (this makes sure that if Agent 2 gives a rating of 4 stars it must talior its review to that rating so we dont see a rating of 1 star and a review of 'this product was wonderfull ')

## importing dependencies

## ── INSTALL FIRST ────────────────────────────────────────────────
!pip install datasets

In [ ]:
import json 
import re
import hashlib
import os
import faiss
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import time
import random
from google import genai
from google.genai import types
from typing import Optional
from enum import Enum
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field, field_validator


from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# use the kaggle secret enviroment to secure my api key 
# put your secret key here or add it from the enviroment here also 

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
HF_Key = UserSecretsClient().get_secret("HF_Key")

In [ ]:
def load_category(
    reviews_path  : str,
    meta_path     : str,
    category_name : str,
    nrows         : int = None    # None = load everything, int = load that many rows
) -> pd.DataFrame:

    """
    Loads and joins reviews + metadata for one Amazon category.
    Returns a clean joined dataframe with standardised column names.

    Parameters:
        reviews_path  : path to reviews .jsonl.gz file
        meta_path     : path to metadata .jsonl.gz file
        category_name : label for this category (stored in source_category)
        nrows         : number of review rows to load — None loads everything
    """
    print(f"Loading {category_name}...")
    if nrows:
        print(f"  Row limit : {nrows:,}")
    else:
        print(f"  Row limit : none — loading full dataset")

    # ── LOAD REVIEWS ──────────────────────────────────────────────
    read_kwargs = {
        'lines'       : True,
        'compression' : 'gzip'
    }
    if nrows is not None:
        read_kwargs['nrows'] = nrows

    reviews_df = pd.read_json(reviews_path, **read_kwargs)
    meta_df    = pd.read_json(meta_path, lines=True, compression='gzip')

    print(f"  Reviews loaded  : {len(reviews_df):,}")
    print(f"  Products loaded : {len(meta_df):,}")

    # ── FLATTEN METADATA LIST FIELDS ─────────────────────────────
    meta_df['description_text'] = meta_df['description'].apply(
        lambda x: ' '.join(x) if isinstance(x, list) and x else ''
    )
    meta_df['features_text'] = meta_df['features'].apply(
        lambda x: ' | '.join(x[:3]) if isinstance(x, list) and x else ''
    )

    # ── SLIM METADATA ─────────────────────────────────────────────
    meta_slim = meta_df[[
        'parent_asin',
        'title',
        'description_text',
        'features_text',
        'price',
        'store',
        'main_category'
    ]].drop_duplicates(subset='parent_asin')

    # ── JOIN ──────────────────────────────────────────────────────
    df = reviews_df.merge(meta_slim, on='parent_asin', how='left')

    # ── STANDARDISE COLUMN NAMES ──────────────────────────────────
    df = df.rename(columns={
        'verified': 'verified_purchase',
        'title_x'          : 'review_title',
        'title_y'          : 'product_title',
    })

    # ── TAG SOURCE ────────────────────────────────────────────────
    df['source_category'] = category_name
    # --Drop Image column---------------
    df = df.drop(columns = "images", axis = 1)
    print(f"  Joined rows     : {len(df):,}")
    return df


In [ ]:
def normalise_video_games_categories(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups fragmented Video Games metadata categories
    into clean top-level buckets for RAG matching.
    Handles null/NaN values gracefully.
    """

    gaming_cats = {
        'video games', 'software', 'toys & games',
        'buy a kindle', 'audible audiobooks'
    }
    electronics_cats = {
        'computers', 'all electronics', 'cell phones & accessories',
        'home audio & theater', 'amazon devices', 'camera & photo',
        'digital music', 'musical instruments', 'portable audio & accessories',
        'car electronics', 'gps & navigation', 'amazon home'
    }
    beauty_cats = {
        'all beauty', 'premium beauty', 'health & personal care',
        'amazon fashion'
    }
    other_cats = {
        'books', 'movies & tv', 'grocery', 'office products',
        'tools & home improvement', 'sports & outdoors',
        'industrial & scientific', 'pet supplies', 'baby',
        'automotive', 'arts, crafts & sewing', 'appliances',
        'collectible coins'
    }

    def map_category(cat):
        # ── HANDLE NULL / NaN / EMPTY ─────────────────────────────
        if cat is None:
            return 'Other'
        if isinstance(cat, float):        # NaN comes in as float
            return 'Other'
        
        cat_lower = str(cat).strip().lower()
        
        if not cat_lower or cat_lower == 'nan':
            return 'Other'
        if cat_lower in gaming_cats      : return 'Video Games and Software'
        if cat_lower in electronics_cats : return 'Electronics'
        if cat_lower in beauty_cats      : return 'Beauty'
        if cat_lower in other_cats       : return 'Other'
        return 'Other'

    # ── APPLY MAPPING ─────────────────────────────────────────────
    df = df.copy()
    df['main_category'] = df['main_category'].apply(map_category)

    # ── VERIFY NO NULLS REMAIN ────────────────────────────────────
    null_count = df['main_category'].isna().sum()
    if null_count > 0:
        print(f"  ⚠️  {null_count} nulls still present — filling with 'Other'")
        df['main_category'] = df['main_category'].fillna('Other')

    # ── FINAL DISTRIBUTION ────────────────────────────────────────
    print("\nCategory distribution after normalisation:")
    dist = df['main_category'].value_counts()
    for cat, count in dist.items():
        print(f"  {str(cat):<35} {count:,}")

    # ── CONFIRM NO NULLS ──────────────────────────────────────────
    remaining_nulls = df['main_category'].isna().sum()
    print(f"\n  Null values remaining : {remaining_nulls}")

    return df

In [ ]:
def merge_categories(
    dataframes       : list[pd.DataFrame],
    drop_duplicates  : bool = True,
    min_text_words   : int  = 10,
    verbose          : bool = True
) -> pd.DataFrame:
    """
    Merges multiple category dataframes into one clean combined dataset.

    Handles:
    - Duplicate reviews (same user reviewing same product across datasets)
    - Column alignment across categories
    - Text quality filter
    - Summary report

    Parameters:
        dataframes      : list of loaded category dataframes
        drop_duplicates : remove same user + same product duplicates
        min_text_words  : minimum words for a review to be kept
        verbose         : print merge summary

    Returns:
        merged_df : single clean combined dataframe
    """

    print("\n" + "=" * 60)
    print("MERGING CATEGORIES")
    print("=" * 60)

    # ── ALIGN COLUMNS ACROSS DATAFRAMES ──────────────────────────
    # get union of all columns
    all_cols = set()
    for df in dataframes:
        all_cols.update(df.columns.tolist())

    # add missing columns as NaN so concat doesn't fail
    aligned = []
    for df in dataframes:
        missing = all_cols - set(df.columns)
        for col in missing:
            df[col] = None
        aligned.append(df)

    # ── CONCAT ───────────────────────────────────────────────────
    merged_df = pd.concat(aligned, ignore_index=True)

    if verbose:
        print(f"\nRaw combined rows : {len(merged_df):,}")
        print(f"Sources           :")
        for cat, count in merged_df['source_category'].value_counts().items():
            print(f"  {cat:<35} {count:,} reviews")

    # ── DROP EXACT DUPLICATES ─────────────────────────────────────
    before = len(merged_df)
    merged_df = merged_df.drop_duplicates()
    if verbose:
        print(f"\nExact duplicates removed  : {before - len(merged_df):,}")

    # ── DROP USER + PRODUCT DUPLICATES ───────────────────────────
    # same user reviewing same product in both datasets
    if drop_duplicates and 'user_id' in merged_df.columns and 'parent_asin' in merged_df.columns:
        before = len(merged_df)
        merged_df = merged_df.drop_duplicates(
            subset  = ['user_id', 'parent_asin'],
            keep    = 'first'
        )
        if verbose:
            print(f"User+product dupes removed: {before - len(merged_df):,}")

    # ── TEXT QUALITY FILTER ───────────────────────────────────────
    if 'text' in merged_df.columns:
        merged_df['text'] = merged_df['text'].fillna('').astype(str)
        before = len(merged_df)
        merged_df = merged_df[
            merged_df['text'].str.split().str.len() >= min_text_words
        ]
        if verbose:
            print(f"Short reviews removed     : {before - len(merged_df):,}")

    # ── RESET INDEX ───────────────────────────────────────────────
    merged_df = merged_df.reset_index(drop=True)

    # ── FINAL SUMMARY ─────────────────────────────────────────────
    if verbose:
        print(f"\n{'=' * 60}")
        print(f"MERGED DATASET READY")
        print(f"{'=' * 60}")
        print(f"  Total rows        : {len(merged_df):,}")
        print(f"  Unique users      : {merged_df['user_id'].nunique():,}")
        print(f"  Unique products   : {merged_df['parent_asin'].nunique():,}")
        print(f"  Categories        : {merged_df['source_category'].nunique()}")

        user_counts = merged_df.groupby('user_id').size()
        viable      = user_counts[user_counts >= 20]
        print(f"  Users with 20+    : {len(viable):,}")
        print(f"  Avg reviews/user  : {user_counts.mean():.1f}")
        print(f"{'=' * 60}\n")

    return merged_df

In [ ]:
# ── DOWNLOAD "All Beauty category dataset" ─────────────────────────────────────────────────────
#----BEWARE DATASET IS HEAVY RUN THIS CELL AT YOUR OWN RISK -------------------------------------
#----THIS USES ALL THE FUNCTIONS ABOVE TO DOWNLOAD, CONCAT AND MERGE THE TWO DATASET TOGETHER----
!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz" \
    -O "/kaggle/working/All_Beauty_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz" \
    -O "/kaggle/working/meta_All_Beauty.jsonl.gz"

# confirm sizes — should be several MB each
!ls -lh /kaggle/working/*.gz

# ── DOWNLOAD video_games CATEGORY ──────────────────────────────────────
# pick whichever category suits your project

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz" \
    -O "/kaggle/working/game_reviews.jsonl.gz"

!wget -q --show-progress \
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz" \
    -O "/kaggle/working/meta_game.jsonl.gz"


# ── LOAD BOTH CATEGORIES ──────────────────────────────────────────

beauty_df = load_category(
    reviews_path  = '/kaggle/working/All_Beauty_reviews.jsonl.gz',
    meta_path     = '/kaggle/working/meta_All_Beauty.jsonl.gz',
    category_name = 'All Beauty',
    nrows         = None         )

# load first 500k rows — for large categories like Books
game_df = load_category(
    reviews_path  = '/kaggle/working/game_reviews.jsonl.gz',
    meta_path     = '/kaggle/working/meta_game.jsonl.gz',
    category_name = 'Books',
    nrows         = 600_000 
    )


# normalise categories
game_df = normalise_video_games_categories(game_df)


# ── MERGE ─────────────────────────────────────────────────────────

df = merge_categories(
dataframes      = [beauty_df, game_df],
drop_duplicates = True,
min_text_words  = 10,
verbose         = True)

In [ ]:
# ── HEALTH CHECK ─────────────────────────────────────────────────
user_counts = df.groupby('user_id').size()
viable      = user_counts[user_counts >= 20]

print("=" * 50)
print("HEALTH CHECK")
print("=" * 50)
print(f"Total reviews         : {len(df):,}")
print(f"Unique users          : {df['user_id'].nunique():,}")
print(f"Unique products       : {df['asin'].nunique():,}")
print(f"Users with 20+ reviews: {len(viable):,}")
print(f"Avg review length     : {df['text'].str.split().str.len().mean():.0f} words")
print(f"Rating distribution   : {df['rating'].value_counts().sort_index().to_dict()}")
#print(f"Missing product title : {df['product_title'].isna().sum():,}")
#print(f"Missing description   : {df['description_text'].isna().sum():,}")
print(f"Verified purchase %   : {df['verified_purchase'].mean()*100:.1f}%")

In [ ]:
df.head()

## Data Exploration 

### Column Description

- Rating: Users provide ratings (on a scale of 1 to 5) for various products.
  
- Title:Concise titles summarizing the content of each review. 
- Text: Detailed textual descriptions of users’ experiences with the products.
- Images: Associated images (if available) related to the reviewed items. 
- ASIN (Amazon Standard Identification Number): Unique identifiers for the products.
- Parent ASIN: Identifiers for parent products (if applicable). 
- User ID: Unique identifiers for the reviewers. 
- Timestamp: Date and time when the review was posted. 
- Helpful Votes: Count of helpful votes received for each review.

- Verified Purchase: Indicates whether the reviewer made a verified purchase. Researchers and practitioners can leverage this dataset for tasks such as sentiment analysis, recommendation systems, and natural language processing. It provides valuable insights into user preferences and product quality on the Amazon platform.

In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

### 1. Checking the Schema first 
Before anything else — lets know exactly what fields we have. Amazon Reviews schema varies by version.

In [ ]:
# understanding the schema of the review dataset 
df.iloc[500].to_dict()

In [ ]:
# checking the columns in the data set 
df.columns.tolist()

### 2.User history Depth 
This is our most critical check. Users with fewer than 20 reviews cannot build a believable persona.

In [ ]:
# Reviews per user

user_counts = df.groupby('user_id').size()
user_counts.describe()

In [ ]:
# How many viable user?

five_viable = user_counts[user_counts >= 5]
ten_viable = user_counts[user_counts >= 10]
twenty_viable = user_counts[user_counts >= 20]

print(f"""The number of users with five reviews or more are {len(five_viable)},
        The number of users with ten reviews or more are {len(ten_viable)},
        while the number of users with twenty reviews or more are {len(twenty_viable)}""")

In [ ]:
# Distribution shape

user_counts.value_counts().head(20)
# expect heavy power law: that is this should be less than the above because above we are asking for greater than or equal to 5 

### 3.Rating Behaviour
we need to understand how users rate — not just globally, but per user. This feeds directly into our persona rating model.

In [ ]:
# Global rating distribution
df['rating'].value_counts().sort_index()

In [ ]:
#Per-user avg rating
df.groupby('user_id')['rating'].mean().describe()

In [ ]:
# Per-user rating variance
df.groupby('user_id')['rating'].std().describe()

In [ ]:
#Rating over time
''' checking how users rate over time that is if a users rating is increasingly critical or generous or indifferent
     does their rating trend change? if so how helps build - persona signal

'''
# Step 1 — sort each user's reviews by time
df = df.sort_values(['user_id', 'timestamp'])

# Step 2 — assign a review sequence number per user (1st review, 2nd review, etc.)
df['review_seq'] = df.groupby('user_id').cumcount() + 1

# Step 3 — for each user, compute the slope of their rating over time
# a positive slope = they rate higher over time
# a negative slope = they become more critical over time
# near zero = consistent rater

def rating_slope(group):
    if len(group) < 5:  # need enough reviews to detect a trend
        return np.nan
    x = group['review_seq'].values
    y = group['rating'].values
    slope = np.polyfit(x, y, 1)[0]  # linear fit, take the slope
    return slope

user_slopes = df.groupby('user_id').apply(rating_slope).reset_index()
user_slopes.columns = ['user_id', 'rating_slope']

# Step 4 — look at the distribution of slopes across all users
print(user_slopes['rating_slope'].describe())

# Step 5 — classify users by their rating trend
def classify_trend(slope):
    if pd.isna(slope):
        return 'insufficient_data'
    elif slope > 0.05:
        return 'increasingly_generous'
    elif slope < -0.05:
        return 'increasingly_critical'
    else:
        return 'consistent'

user_slopes['trend'] = user_slopes['rating_slope'].apply(classify_trend)
print(user_slopes['trend'].value_counts())

In [ ]:
# inspect one specific user's rating journey
sample_user = df[df['user_id'] == df['user_id'].iloc[129]]
display(sample_user[['timestamp', 'rating','review_seq', 'text']].to_string())

In [ ]:
# Check a 30 reviews by myself to see if whats there is really material to build a case file on a reviewer 

import textwrap

def profile_user(df, user_id=None, min_reviews=30):
    """
    Pull one user's full review history and print it as a readable story.
    If no user_id given, randomly picks a user with min_reviews+ reviews.
    """
    
    # filter to viable users
    user_counts = df.groupby('user_id').size()
    viable_users = user_counts[user_counts >= min_reviews].index.tolist()
    
    if len(viable_users) == 0:
        print(f"No users with {min_reviews}+ reviews found.")
        return None
    
    # pick a user
    if user_id is None:
        user_id = pd.Series(viable_users).sample(1).iloc[0]
        print(f"Randomly selected user: {user_id}\n")
    elif user_id not in viable_users:
        actual_count = user_counts.get(user_id, 0)
        print(f"User {user_id} only has {actual_count} reviews. Try another.")
        return None

    # pull their reviews, sorted by time
    user_df = df[df['user_id'] == user_id].sort_values('timestamp').reset_index(drop=True)
    
    # ── HEADER ──────────────────────────────────────────────
    print("=" * 65)
    print(f"USER PROFILE: {user_id}")
    print("=" * 65)
    
    # ── QUICK STATS ─────────────────────────────────────────
    avg_rating   = user_df['rating'].mean()
    rating_std   = user_df['rating'].std()
    avg_length   = user_df['text'].str.split().str.len().mean()
    rating_dist  = user_df['rating'].value_counts().sort_index().to_dict()
    
    print(f"\n📊 QUICK STATS")
    print(f"   Total reviews   : {len(user_df)}")
    print(f"   Avg rating      : {avg_rating:.2f} ⭐  (std: {rating_std:.2f})")
    print(f"   Avg review length: {avg_length:.0f} words")
    print(f"   Rating breakdown: {rating_dist}")
    
    # ── RATING TREND ────────────────────────────────────────
    first_half = user_df.iloc[:len(user_df)//2]['rating'].mean()
    second_half = user_df.iloc[len(user_df)//2:]['rating'].mean()
    trend = "↑ more generous over time" if second_half > first_half + 0.2 \
            else "↓ more critical over time" if second_half < first_half - 0.2 \
            else "→ consistent rater"
    print(f"   Rating trend    : {trend}  (early avg: {first_half:.2f} → late avg: {second_half:.2f})")
    
    # ── CATEGORIES ──────────────────────────────────────────
    if 'category' in user_df.columns:
        top_cats = user_df['category'].value_counts().head(5).to_dict()
        print(f"\n🗂  TOP CATEGORIES")
        for cat, count in top_cats.items():
            print(f"   {cat:<35} {count} reviews")
    
    # ── FULL REVIEW HISTORY ─────────────────────────────────
    print(f"\n📝 FULL REVIEW HISTORY  (oldest → newest)")
    print("-" * 65)
    
    for i, row in user_df.iterrows():
        # convert unix timestamp to readable date
        try:
            date = pd.to_datetime(row['timestamp'], unit='s').strftime('%b %Y')
        except:
            date = "Unknown date"
        
        stars     = "⭐" * int(row['rating'])
        title   = row.get('title', '')
        review    = row.get('text', '')
        word_count = len(str(review).split())
        
        print(f"\n[{i+1}]  {date}  |  {stars} ({int(row['rating'])}/5)  |  {word_count} words")
        
        if title:
            print(f"     Headline : {title}")
        
        # wrap the review text so it's readable
        wrapped = textwrap.fill(str(review), width=60, initial_indent="     ", subsequent_indent="     ")
        print(wrapped)
        print()
    
    print("=" * 65)
    print("ASK YOURSELF:")
    print("  1. Can I describe this person in 3 sentences?")
    print("  2. Do they have a consistent complaint or praise pattern?")
    print("  3. Does their writing style feel distinctive?")
    print("  4. Would I recognise their review if I saw it without a name?")
    print("=" * 65)
    
    return user_df

In [ ]:
"""
if you go through this review below you will see that you should be able to build a case file on the reviewer 
eg shes a woman, she loves body products , lives in the desert area , has a dry skin problem ,
doesnt give 5 easily, gives 4 mostly rating wise 
"""
# random user with 30+ reviews
#user_history = profile_user(df)
# or target a specific user you already found
user_history = profile_user(df, user_id='AHBWH2LBU3NFLD46GKJKIBAHKXEQ')

# or lower the bar for testing
#user_history = profile_user(df, min_reviews=20)

In [ ]:
# random user with 30+ reviews
user_history = profile_user(df)

### 4. Review text signals
The text is where persona lives. we need to know if there's enough text per user to extract writing style.

# Review length distribution
df['text'].str.split().str.len().describe()

In [ ]:
#Empty / very short reviews
# we have to filter this out cause the review is too short

df[df['text'].str.len() < 20].shape[0]

In [ ]:
# Avg text length per user (BEWARE computationaly expensive to run )
#avg_text_per_user = df.groupby('user_id')['text'].apply(lambda x: x.str.len().mean())


In [ ]:
#display(avg_text_per_user)

In [ ]:
#print(avg_text_per_user.value_counts())

In [ ]:
# Verified purchase flag

df['verified_purchase'].value_counts()

## Data preprocessing
### Combined function to get the rich-review data
this function uses the data exploration techniques above to get the rich reviews and also cleans the data

In [ ]:
def clean_amazon_reviews(df, 
                          min_reviews=10, 
                          min_word_count=30,
                          verified_only=True):
    """
    Full cleaning pipeline for Amazon Reviews dataset.
    Returns a clean dataframe of viable users ready for persona building.
    """
    
    print("=" * 60)
    print("AMAZON REVIEWS — CLEANING PIPELINE")
    print("=" * 60)
    print(f"\n▶ Starting shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # ── STEP 1: DROP DUPLICATE ROWS ─────────────────────────
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"\n[1] Duplicate rows removed   : {before - after:,}")
    print(f"    Remaining                : {after:,}")
    
    # ── STEP 2: DROP DUPLICATE COLUMNS ──────────────────────
    before_cols = df.shape[1]
    df = df.loc[:, ~df.columns.duplicated()]
    after_cols = df.shape[1]
    print(f"\n[2] Duplicate columns removed: {before_cols - after_cols}")
    print(f"    Remaining columns        : {list(df.columns)}")
    
    # ── STEP 3: REQUIRE CRITICAL COLUMNS ────────────────────
    critical_cols = ['user_id', 'rating', 'text']
    missing_cols  = [c for c in critical_cols if c not in df.columns]
    
    if missing_cols:
        print(f"\n❌ STOPPING — missing critical columns: {missing_cols}")
        print(f"   Your dataset has: {list(df.columns)}")
        return None
    
    before = len(df)
    df = df.dropna(subset=critical_cols)
    after = len(df)
    print(f"\n[3] Rows dropped (missing critical fields): {before - after:,}")
    print(f"    Remaining                              : {after:,}")
    
    # ── STEP 4: CLEAN REVIEW TEXT ───────────────────────────
    # strip whitespace
    df['text'] = df['text'].astype(str).str.strip()
    
    # remove placeholder text that slips through
    placeholders = ['n/a', 'na', 'none', 'null', '.', '-', 'no review']
    before = len(df)
    df = df[~df['text'].str.lower().isin(placeholders)]
    
    # remove very short reviews (below min_word_count)
    df['word_count'] = df['text'].str.split().str.len()
    df = df[df['word_count'] >= min_word_count]
    after = len(df)
    print(f"\n[4] Rows dropped (empty / short reviews) : {before - after:,}")
    print(f"    Min word count threshold             : {min_word_count} words")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 5: VERIFIED PURCHASES ONLY ─────────────────────
    if verified_only and 'verified_purchase' in df.columns:
        before = len(df)
        df = df[df['verified_purchase'] == True]
        after = len(df)
        print(f"\n[5] Rows dropped (unverified purchases)  : {before - after:,}")
        print(f"    Remaining                            : {after:,}")
    elif 'verified_purchase' not in df.columns:
        print(f"\n[5] Skipped — 'verified' column not found in dataset")
    
    # ── STEP 6: VALID RATINGS ONLY ──────────────────────────
    before = len(df)
    df = df[df['rating'].between(1, 5)]
    after = len(df)
    print(f"\n[6] Rows dropped (invalid ratings)       : {before - after:,}")
    print(f"    Remaining                            : {after:,}")
    
    # ── STEP 7: FILTER TO USERS WITH 20+ REVIEWS ────────────
    before = len(df)
    user_counts  = df.groupby('user_id').size()
    viable_users = user_counts[ user_counts >= min_reviews].index
    df = df[df['user_id'].isin(viable_users)]
    after = len(df)
    dropped_users = len(user_counts) - len(viable_users)
    print(f"\n[7] Users dropped (< {min_reviews} reviews)          : {dropped_users:,}")
    print(f"    Viable users remaining      user_counts         : {len(viable_users):,}")
    print(f"    Rows remaining                       : {after:,}")
    
    # ── STEP 8: SORT BY USER + TIME ─────────────────────────
    if 'timestamp' in df.columns:
        df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)
        print(f"\n[8] Sorted by user_id + timestamp ✓")
    else:
        df = df.sort_values('user_id').reset_index(drop=True)
        print(f"\n[8] Sorted by user_id (no timestamp found) ✓")
    
    # ── FINAL SUMMARY ───────────────────────────────────────
    print(f"\n{'=' * 60}")
    print(f"✅ CLEAN DATASET READY")
    print(f"{'=' * 60}")
    print(f"   Final shape      : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Unique users     : {df['user_id'].nunique():,}")
    print(f"   Avg reviews/user : {df.groupby('user_id').size().mean():.1f}")
    print(f"   Avg word count   : {df['word_count'].mean():.0f} words")
    print(f"   Rating breakdown : {df['rating'].value_counts().sort_index().to_dict()}")
    print(f"{'=' * 60}\n")
    
    return df

In [ ]:
rich_df = clean_amazon_reviews(df,min_reviews=10)

In [ ]:
rich_df.head()

In [ ]:
rich_df.to_csv('rich_users.csv', index=False)